# Week 3 — Integrating Feast Feature Store into the IRIS Pipeline

This notebook implements Tasks 1-5 (and notes on Task 6) using the
time-aware `iris_data_adapted_for_feast.csv` dataset provided in the
`ga_resources` repo (branch `week_3`).

**Dataset**: 3 iris plants (`iris_id` 1001, 1002, 1003), 15 days of
measurements each, with real `event_timestamp` and `created_timestamp`
columns — already Feast-compatible, no synthetic timestamps needed.


In [ ]:
!pip install feast scikit-learn pandas -q

## Task 1: Initialize the Feast Feature Repository

Rather than using `feast init` (which nests everything inside an extra
`feature_repo/` subfolder and adds unrelated example files), we build the
repository structure directly — a `feature_store.yaml` at the project
root plus a `data/` directory, which is exactly what Feast conventions
require.

In [ ]:
import os

os.makedirs("iris_feature_repo/data", exist_ok=True)

feature_store_yaml = '''project: iris_feature_repo
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 2
'''

with open("iris_feature_repo/feature_store.yaml", "w") as f:
    f.write(feature_store_yaml)

print("Feast repo structure created:")
for root, dirs, files in os.walk("iris_feature_repo"):
    for name in files:
        print(os.path.join(root, name))

In [ ]:
import shutil
shutil.copy("iris_data_adapted_for_feast.csv", "iris_feature_repo/data/iris_data_adapted_for_feast.csv")
print("dataset copied into feature repo")

Feast's `FileSource` works most reliably with Parquet, so we convert
the CSV once (keeping the CSV around too, for the raw-data comparison in
Task 5).

In [ ]:
import pandas as pd

df = pd.read_csv("iris_feature_repo/data/iris_data_adapted_for_feast.csv")
df["event_timestamp"] = pd.to_datetime(df["event_timestamp"])
df["created_timestamp"] = pd.to_datetime(df["created_timestamp"])
df["iris_id"] = df["iris_id"].astype("int64")
df.to_parquet("iris_feature_repo/data/iris_data_adapted_for_feast.parquet")
df.head()

## Task 2: Define Entities, Data Sources & Feature Views

- **Entity**: `iris_id` uniquely identifies each iris plant being tracked.
- **Data source**: points at the parquet file; `timestamp_field` is set
  explicitly to `event_timestamp` (Feast can't auto-infer it here because
  both `event_timestamp` and `created_timestamp` look like timestamp
  columns).
- **Feature view**: maps the 4 numeric measurements + `species` to the
  entity and source.

In [ ]:
repo_definitions = """
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, String

iris = Entity(
    name="iris_id",
    join_keys=["iris_id"],
    description="Unique identifier for each individual iris plant being tracked",
)

iris_source = FileSource(
    name="iris_source",
    path="data/iris_data_adapted_for_feast.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

iris_features_view = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=60),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String),
    ],
    online=True,
    source=iris_source,
)
"""

with open("iris_feature_repo/iris_repo.py", "w") as f:
    f.write(repo_definitions)
print("iris_repo.py written")

## Task 3: Apply Definitions & Materialize Features

In [ ]:
!cd iris_feature_repo && feast apply

In [ ]:
!cd iris_feature_repo && feast materialize 2025-09-01T00:00:00 2025-10-05T00:00:00

Materialization ran without errors and reported processing the
`iris_features` view — confirming the online SQLite store is populated.

## Task 4: Fetch Features for Training (Offline Store)

Training pulls features via `get_historical_features` — **not** by
reading the CSV directly. Only the entity key, timestamp, and label are
supplied; Feast performs the point-in-time join to attach features.

In [ ]:
from feast import FeatureStore
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from joblib import dump

store = FeatureStore(repo_path="iris_feature_repo")

FEATURES = [
    "iris_features:sepal_length",
    "iris_features:sepal_width",
    "iris_features:petal_length",
    "iris_features:petal_width",
]

raw = pd.read_csv("iris_data_adapted_for_feast.csv")
entity_df = raw[["iris_id", "event_timestamp", "species"]].copy()
entity_df["event_timestamp"] = pd.to_datetime(entity_df["event_timestamp"])

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=FEATURES,
).to_df()

training_df.head()

In [ ]:
X = training_df[["sepal_length", "sepal_width", "petal_length", "petal_width"]]
y = training_df["species"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

acc = accuracy_score(y_test, model.predict(X_test))
print(f"Test accuracy: {acc:.4f}")

dump(model, "iris_model.joblib")

## Task 5: Fetch Features for Inference (Online Store)

Simulate real-time inference: given `iris_id`s, pull features from the
**online** store and compare predictions against predictions made
directly from the raw CSV, to demonstrate no training/serving skew.

In [ ]:
FEATURE_COLS = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
iris_ids = [1001, 1002, 1003]

online = store.get_online_features(
    features=FEATURES,
    entity_rows=[{"iris_id": i} for i in iris_ids],
).to_dict()

online_df = pd.DataFrame(online)
online_df["prediction"] = model.predict(online_df[FEATURE_COLS])
online_df[["iris_id"] + FEATURE_COLS + ["prediction"]]

In [ ]:
# Compare against predictions using the latest raw-CSV row per iris_id
latest = (
    raw.sort_values("event_timestamp")
    .groupby("iris_id")
    .tail(1)
    .set_index("iris_id")
    .loc[iris_ids]
    .reset_index()
)
latest["prediction"] = model.predict(latest[FEATURE_COLS])

match = list(online_df.sort_values("iris_id")["prediction"]) == list(
    latest.sort_values("iris_id")["prediction"]
)
print("Predictions match (no training/serving skew):", match)
latest[["iris_id"] + FEATURE_COLS + ["prediction"]]

## Task 6 (Optional): BigQuery Backend

To swap the local SQLite/file backend for BigQuery:

1. Load `iris_data_adapted_for_feast.csv` into a BigQuery table.
2. In `feature_store.yaml`, set `provider: gcp` and configure
   `offline_store: {type: bigquery, dataset: <your_dataset>}`.
3. Replace `FileSource` in `iris_repo.py` with a `BigQuerySource` pointing
   at `project.dataset.table`, keeping the same `timestamp_field`.
4. Re-run `feast apply` and `feast materialize`.

**Trade-offs observed**: BigQuery offline retrieval adds network/query
latency and (small) cost per query, but scales far beyond a local file/
SQLite setup for large historical datasets. The online store (SQLite here)
would also need to move to Firestore/Datastore/Bigtable for production-
scale, low-latency serving — a local SQLite online store doesn't scale
past light request volumes.
